# Astrisk model: fine-tune on the classical books and engine charts

Runs on a **free GPU**: Google Colab (Runtime → Change runtime type → T4 GPU) or Kaggle (Accelerator → GPU T4).

1. On the server run `scripts/pack-llm-data.sh`; it writes `.runtime/llm-data/astrisk-train.zip`.
2. Upload `astrisk-train.zip` here (Colab: the folder icon on the left; Kaggle: Add data → Upload).
3. Run all cells. About 1.5–2.5 hours on a T4.
4. Download `astrisk-qwen3-1.7b-Q8_0.gguf` at the end and copy it to `~/panchang/.runtime/llm/` on the server.

Stage 1 reads the public-domain books themselves (Brihat Jataka in three translations, Brihat Samhita in two).
Stage 2 teaches the answer format: plain answer first, the book's view with its verse, chart details last,
using questions on every checked book rule and on charts computed by the Astrisk engine.
The base model is Qwen3-1.7B (Apache-2.0).

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --upgrade trl datasets

In [ ]:
import glob, json, os, random, zipfile
zips = glob.glob('/content/astrisk-train.zip') + glob.glob('/kaggle/input/**/astrisk-train.zip', recursive=True) + glob.glob('astrisk-train.zip')
assert zips, 'Upload astrisk-train.zip first'
os.makedirs('data', exist_ok=True)
zipfile.ZipFile(zips[0]).extractall('data')
read = lambda f: [json.loads(l) for l in open('data/' + f)]
books, book_qa, train, bench = read('books.jsonl'), read('book-qa.jsonl'), read('train.jsonl'), read('book-bench.jsonl')
predict_train, predict_test = read('predict-train.jsonl'), read('predict-test.jsonl')
print(len(books), 'book passages;', len(book_qa), 'book Q&A;', len(train), 'chart examples;', len(predict_train), 'prediction examples;', len(bench), 'benchmark items;', len(predict_test), 'prediction test items')

In [ ]:
from unsloth import FastLanguageModel
import torch
MAX_LEN = 4096
model, tokenizer = FastLanguageModel.from_pretrained('unsloth/Qwen3-1.7B', max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=7)

## Stage 1: read the books

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def cite(p):
    who = f"tr. {p['translator']}, " if p.get('translator') else ''
    return f"{p['title']} ({who}{p['year']}) {p['ref']}\n{p['text']}" + tokenizer.eos_token

book_ds = Dataset.from_list([{'text': cite(p)} for p in books])
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=book_ds,
    args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_LEN, packing=True, per_device_train_batch_size=2,
                   gradient_accumulation_steps=4, num_train_epochs=1, learning_rate=1e-4, warmup_ratio=0.03,
                   lr_scheduler_type='cosine', logging_steps=10, optim='adamw_8bit', output_dir='out-books', report_to='none', seed=7))
trainer.train()

## Stage 2: answer like Astrisk

In [ ]:
random.seed(7)
def fits(ex):
    return len(tokenizer.apply_chat_template(ex['messages'], tokenize=True, enable_thinking=False)) <= MAX_LEN

chat = [e for e in train if e['meta']['task'] == 'chat']
readings = [e for e in train if e['meta']['task'] == 'reading']
random.shuffle(chat); random.shuffle(readings); random.shuffle(predict_train)
mix = book_qa * 2 + chat[:2500] + readings[:150] + predict_train[:2000]
mix = [e for e in mix if fits(e)]
random.shuffle(mix)
sft_ds = Dataset.from_list([{'text': tokenizer.apply_chat_template(e['messages'], tokenize=False, enable_thinking=False)} for e in mix])
print(len(sft_ds), 'training conversations')

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=sft_ds,
    args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_LEN, packing=False, per_device_train_batch_size=1,
                   gradient_accumulation_steps=8, num_train_epochs=1, learning_rate=1e-4, warmup_ratio=0.03,
                   lr_scheduler_type='cosine', logging_steps=10, optim='adamw_8bit', output_dir='out-sft', report_to='none', seed=7))
# Learn only the answers, not the prompts.
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train()

## Check exact answers on the book benchmark

Each item has the verse and the plain answer the model must give. An answer passes when it cites the right verse and covers the book's meaning.

In [ ]:
import re
FastLanguageModel.for_inference(model)
SYSTEM = 'You are Astrisk. Answer in plain language and cite the classical source. Return JSON only: {"answer": string}.'
def ask(q, max_new=350):
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': q}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to('cuda')
    out = model.generate(ids, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

words = lambda s: set(w for w in re.findall(r'[a-z]+', s.lower()) if len(w) > 3)
random.seed(1)
sample = random.sample(bench, 60)
passed = 0
for it in sample:
    a = ask(it['question'])
    ok_ref = it['ref'] in a
    overlap = len(words(it['plain']) & words(a)) / max(1, len(words(it['plain'])))
    passed += ok_ref and overlap >= 0.5
print(f'book benchmark: {passed}/{len(sample)} exact (right verse and the book\'s meaning)')
print(ask(sample[0]['question']))

## Check predictions on the prediction test set

Charts the model never saw. Each item gives the chart, the timing (or match) lines and a question; the answer must state the right periods, dates, houses and scores, and never promise events. The app's own answers score 100% here (llmdata predict), so this measures the model, not the grader.

In [ ]:
import collections
FORBIDDEN = re.compile(r"(?i)\b(you will (die|divorce|get divorced|fall ill|lose (all|everything))|death is|early death|guaranteed|certainly will|100% sure|will definitely)\b")
SENT = re.compile(r"[.!?\n]\s+")
def grade(it, a):
    failed = []
    for c in it['checks']:
        text = a
        if c.get('within'):
            w = re.compile('(?i)' + c['within'])
            text = ' | '.join(s for s in SENT.split(a) if w.search(s))
        ok = not c.get('any') or any(re.search('(?i)' + p, text) for p in c['any'])
        if any(re.search('(?i)' + p, text) for p in c.get('not', [])):
            ok = False
        if not ok:
            failed.append(c['name'])
    n = len(it['checks'])
    if FORBIDDEN.search(a):
        failed.append('forbidden_wording'); n += 1
    return (n - len(failed)) / max(n, 1), failed

def ask_chat(system, prompt, max_new=450):
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to('cuda')
    out = model.generate(ids, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

random.seed(2)
sample = random.sample(predict_test, 80)
by_kind = collections.defaultdict(list)
for it in sample:
    score, failed = grade(it, ask_chat(it['system'], it['prompt']))
    by_kind[it['kind']].append(score)
total = sum(sum(v) for v in by_kind.values()) / len(sample)
print(f'prediction test: {total:.0%} of facts right on {len(sample)} items')
for k, v in sorted(by_kind.items()):
    print(f'  {k}: {sum(v)/len(v):.0%} of {len(v)}')

## Export for the Astrisk server

In [ ]:
model.save_pretrained_gguf('astrisk-qwen3-1.7b', tokenizer, quantization_method='q8_0')
import glob; print(glob.glob('astrisk-qwen3-1.7b*/*.gguf') + glob.glob('*.gguf'))

In [ ]:
# Colab: download the file. Kaggle: it is under /kaggle/working (Output tab).
try:
    from google.colab import files
    f = (glob.glob('astrisk-qwen3-1.7b*/*.gguf') + glob.glob('*.gguf'))[0]
    os.rename(f, 'astrisk-qwen3-1.7b-Q8_0.gguf')
    files.download('astrisk-qwen3-1.7b-Q8_0.gguf')
except ImportError:
    pass